# Teachable Machine — CNN

Herramienta interactiva para entrenar y probar modelos de clasificación de imágenes.

**Flujo:**
1. Cargar datos (apuntar a carpeta con subcarpetas por clase)
2. Entrenar modelo (un clic) o cargar uno existente (.h5)
3. Probar con imágenes
4. Exportar modelo y configuración

> Ejecute cada celda en orden. Los botones hacen el trabajo.

In [ ]:
import os, glob, random, shutil, io
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import (
    ImageDataGenerator, load_img, img_to_array
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from sklearn.metrics import classification_report, confusion_matrix
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML as IHTML

random.seed(42); np.random.seed(42); tf.random.set_seed(42)

# Estado global compartido entre todas las celdas
_S = {
    'model': None, 'class_names': [], 'img_size': 128,
    'train_gen': None, 'val_gen': None, 'history': None,
    'dataset_dir': '', 'split_dir': ''
}

print(f"TensorFlow {tf.__version__} — Listo")

---
## 1. Cargar Datos

Apunte a una carpeta con subcarpetas por clase (ej: `data/tapitas/rojo/`, `data/tapitas/azul/`).
Las clases se detectan automáticamente.

In [ ]:
w_data_path = widgets.Text(
    value='../data/tapitas', placeholder='Ruta a carpeta con clases',
    description='Dataset:', layout=widgets.Layout(width='500px')
)
w_imgsize = widgets.Dropdown(
    options=[64, 128, 224], value=128, description='Resoluci\u00f3n:'
)
w_btn_data = widgets.Button(
    description=' Cargar datos', button_style='primary', icon='folder-open'
)
w_out_data = widgets.Output()

def _cargar_datos(_):
    with w_out_data:
        clear_output(wait=True)
        ddir = w_data_path.value.strip()
        if not os.path.isdir(ddir):
            print(f"\u2718 No se encontr\u00f3: {ddir}")
            return

        # Detectar clases
        clases = sorted([d for d in os.listdir(ddir)
                         if os.path.isdir(os.path.join(ddir, d))])
        if len(clases) < 2:
            print(f"\u2718 Se necesitan al menos 2 subcarpetas (clases). Encontradas: {clases}")
            return

        img_size = w_imgsize.value
        counts = {}
        for c in clases:
            imgs = [f for f in os.listdir(os.path.join(ddir, c))
                    if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
            counts[c] = len(imgs)

        total = sum(counts.values())
        _S['dataset_dir'] = ddir
        _S['class_names'] = clases
        _S['img_size'] = img_size

        # Mostrar resumen
        print(f"\u2714 Dataset: {ddir}")
        print(f"  Clases: {clases}")
        print(f"  Resoluci\u00f3n: {img_size}\u00d7{img_size}")
        print(f"  Total: {total} im\u00e1genes\n")
        for c, n in counts.items():
            bar = '\u2588' * max(1, n * 30 // max(counts.values()))
            print(f"  {c:>15s} {bar} {n}")

        # Preview
        n_cols = min(4, len(clases))
        fig, axes = plt.subplots(1, n_cols, figsize=(4 * n_cols, 4))
        if n_cols == 1:
            axes = [axes]
        for ax, c in zip(axes, clases[:n_cols]):
            cdir = os.path.join(ddir, c)
            imgs = [f for f in os.listdir(cdir)
                    if f.lower().endswith(('.jpg', '.jpeg', '.png'))]
            if imgs:
                img = load_img(os.path.join(cdir, random.choice(imgs)),
                               target_size=(img_size, img_size))
                ax.imshow(img)
            ax.set_title(f'{c} ({counts[c]})', fontsize=12, fontweight='bold')
            ax.axis('off')
        plt.suptitle('Vista previa por clase', fontsize=14)
        plt.tight_layout()
        plt.show()

w_btn_data.on_click(_cargar_datos)
display(widgets.VBox([w_data_path, w_imgsize, w_btn_data, w_out_data]))

---
## 2. Entrenar Modelo

Entrene un modelo nuevo con un clic, o cargue uno existente (.h5).

**Opciones avanzadas** (opcionales): ajuste augmentation y arquitectura.

In [ ]:
# --- Pesta\u00f1a: Entrenar nuevo vs Cargar existente ---

# == Controles de entrenamiento ==
w_epochs = widgets.IntSlider(value=30, min=5, max=100, step=5,
                             description='\u00c9pocas m\u00e1x:')
w_lr = widgets.FloatLogSlider(value=0.001, base=10, min=-4, max=-2, step=0.5,
                              description='Learning rate:')

# Augmentation
w_rotation = widgets.IntSlider(value=20, min=0, max=90, step=5,
                               description='Rotaci\u00f3n:')
w_shift = widgets.FloatSlider(value=0.1, min=0, max=0.3, step=0.05,
                              description='Desplazamiento:')
w_brightness_lo = widgets.FloatSlider(value=0.8, min=0.5, max=1.0, step=0.05,
                                      description='Brillo m\u00edn:')
w_brightness_hi = widgets.FloatSlider(value=1.2, min=1.0, max=1.5, step=0.05,
                                      description='Brillo m\u00e1x:')
w_zoom = widgets.FloatSlider(value=0.1, min=0, max=0.3, step=0.05,
                             description='Zoom:')

w_aug_accordion = widgets.Accordion(
    children=[widgets.VBox([w_rotation, w_shift, w_brightness_lo,
                            w_brightness_hi, w_zoom])]
)
w_aug_accordion.set_title(0, 'Data Augmentation (ajustar)')
w_aug_accordion.selected_index = None  # Cerrado por defecto

w_train_accordion = widgets.Accordion(
    children=[widgets.VBox([w_epochs, w_lr])]
)
w_train_accordion.set_title(0, 'Hiperpar\u00e1metros (ajustar)')
w_train_accordion.selected_index = None

w_btn_train = widgets.Button(
    description=' Entrenar modelo', button_style='success', icon='play',
    layout=widgets.Layout(width='200px', height='40px')
)
w_progress = widgets.IntProgress(value=0, min=0, max=100,
                                  description='Progreso:',
                                  layout=widgets.Layout(width='500px'))
w_progress.layout.visibility = 'hidden'

# == Cargar modelo existente ==
w_upload_h5 = widgets.FileUpload(accept='.h5,.keras', multiple=False,
                                 description='Subir .h5')
w_path_h5 = widgets.Text(value='', placeholder='O ruta al archivo .h5',
                         description='Ruta:',
                         layout=widgets.Layout(width='500px'))
w_btn_load = widgets.Button(
    description=' Cargar modelo', button_style='info', icon='upload',
    layout=widgets.Layout(width='200px', height='40px')
)

w_out_train = widgets.Output()

# --- L\u00f3gica de entrenamiento ---
class _EpochLogger(tf.keras.callbacks.Callback):
    def __init__(self, progress, output, max_epochs):
        self.progress = progress
        self.output = output
        self.max_epochs = max_epochs
    def on_epoch_end(self, epoch, logs=None):
        self.progress.value = int((epoch + 1) / self.max_epochs * 100)
        with self.output:
            acc = logs.get('accuracy', 0)
            val_acc = logs.get('val_accuracy', 0)
            print(f"  \u00c9poca {epoch+1}: train={acc:.1%} val={val_acc:.1%}",
                  flush=True)

def _split_dataset(ddir, split_dir, class_names):
    """Divide dataset en train/validation 80/20."""
    train_dir = os.path.join(split_dir, 'train')
    val_dir = os.path.join(split_dir, 'validation')
    if os.path.exists(train_dir) and os.path.exists(val_dir):
        return
    random.seed(42)
    for clase in class_names:
        imgs = sorted([f for f in os.listdir(os.path.join(ddir, clase))
                       if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        random.shuffle(imgs)
        n_train = int(len(imgs) * 0.8)
        for subset, img_list in [('train', imgs[:n_train]),
                                  ('validation', imgs[n_train:])]:
            dest = os.path.join(split_dir, subset, clase)
            os.makedirs(dest, exist_ok=True)
            for img in img_list:
                src = os.path.join(ddir, clase, img)
                dst = os.path.join(dest, img)
                if not os.path.exists(dst):
                    shutil.copy(src, dst)

def _build_model(img_size, num_classes):
    """Crea CNN simple de 3 bloques."""
    inputs = layers.Input(shape=(img_size, img_size, 3))
    x = inputs
    for filters in [32, 64, 128]:
        x = layers.Conv2D(filters, 3, padding='same', activation='relu')(x)
        x = layers.BatchNormalization()(x)
        x = layers.MaxPooling2D(2)(x)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.5)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)
    return models.Model(inputs=inputs, outputs=outputs)

def _entrenar(_):
    with w_out_train:
        clear_output(wait=True)
        if not _S['class_names']:
            print("\u2718 Primero cargue los datos (Paso 1).")
            return

        ddir = _S['dataset_dir']
        class_names = _S['class_names']
        img_size = _S['img_size']
        num_classes = len(class_names)
        epochs = w_epochs.value
        lr = w_lr.value

        # Split
        split_dir = ddir + '_split'
        _S['split_dir'] = split_dir
        print("Preparando datos...")
        _split_dataset(ddir, split_dir, class_names)

        # Generators
        train_datagen = ImageDataGenerator(
            rescale=1./255,
            rotation_range=w_rotation.value,
            width_shift_range=w_shift.value,
            height_shift_range=w_shift.value,
            brightness_range=[w_brightness_lo.value, w_brightness_hi.value],
            zoom_range=w_zoom.value,
            fill_mode='nearest'
        )
        val_datagen = ImageDataGenerator(rescale=1./255)

        train_gen = train_datagen.flow_from_directory(
            os.path.join(split_dir, 'train'),
            target_size=(img_size, img_size),
            batch_size=32, class_mode='categorical', shuffle=True
        )
        val_gen = val_datagen.flow_from_directory(
            os.path.join(split_dir, 'validation'),
            target_size=(img_size, img_size),
            batch_size=32, class_mode='categorical', shuffle=False
        )
        _S['train_gen'] = train_gen
        _S['val_gen'] = val_gen

        # Build & train
        print(f"Construyendo CNN ({img_size}px, {num_classes} clases)...")
        model = _build_model(img_size, num_classes)
        model.compile(optimizer=Adam(learning_rate=lr),
                      loss='categorical_crossentropy', metrics=['accuracy'])

        w_progress.value = 0
        w_progress.layout.visibility = 'visible'

        print(f"\nEntrenando ({train_gen.samples} train, {val_gen.samples} val)...\n")
        callbacks = [
            EarlyStopping(patience=10, restore_best_weights=True, verbose=1),
            ReduceLROnPlateau(factor=0.5, patience=5, verbose=1),
            _EpochLogger(w_progress, w_out_train, epochs)
        ]

        history = model.fit(
            train_gen, epochs=epochs, validation_data=val_gen,
            callbacks=callbacks, verbose=0
        )
        _S['model'] = model
        _S['history'] = history
        w_progress.value = 100

        # Resultados
        val_acc = history.history['val_accuracy'][-1]
        train_acc = history.history['accuracy'][-1]
        n_epochs = len(history.history['accuracy'])

        print(f"\n\u2714 Entrenamiento terminado en {n_epochs} \u00e9pocas")
        print(f"  Accuracy entrenamiento: {train_acc:.1%}")
        print(f"  Accuracy validaci\u00f3n:    {val_acc:.1%}")

        gap = train_acc - val_acc
        if gap > 0.15:
            print(f"  \u26a0 Gap de {gap:.1%} — posible overfitting")

        # Curvas
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
        ep = range(1, n_epochs + 1)
        ax1.plot(ep, history.history['accuracy'], 'b-', label='Train')
        ax1.plot(ep, history.history['val_accuracy'], 'r-', label='Val')
        ax1.set_title('Accuracy'); ax1.legend(); ax1.grid(alpha=0.3)
        ax2.plot(ep, history.history['loss'], 'b-', label='Train')
        ax2.plot(ep, history.history['val_loss'], 'r-', label='Val')
        ax2.set_title('Loss'); ax2.legend(); ax2.grid(alpha=0.3)
        plt.tight_layout()
        plt.show()

def _cargar_h5(_):
    with w_out_train:
        clear_output(wait=True)
        # Obtener ruta
        model_path = None
        if w_upload_h5.value:
            info = w_upload_h5.value
            if isinstance(info, dict):
                fname = list(info.keys())[0]
                content = info[fname]['content']
            else:
                fname = info[0].name
                content = info[0].content
            model_path = f'/tmp/{fname}'
            with open(model_path, 'wb') as f:
                f.write(content)
        elif w_path_h5.value.strip():
            model_path = w_path_h5.value.strip()
        else:
            print("\u2718 Suba un archivo .h5 o escriba la ruta.")
            return

        if not os.path.exists(model_path):
            print(f"\u2718 No existe: {model_path}")
            return

        try:
            model = load_model(model_path, compile=False)
            model.compile(optimizer='adam', loss='categorical_crossentropy',
                          metrics=['accuracy'])
        except Exception as e:
            print(f"\u2718 Error: {e}")
            return

        # Auto-detectar img_size del modelo
        detected_size = model.input_shape[1]
        if detected_size and detected_size != _S['img_size']:
            print(f"  Ajustando resoluci\u00f3n a {detected_size} (detectada del modelo)")
            _S['img_size'] = detected_size

        # Auto-detectar num_classes
        n_out = model.output_shape[-1]
        if _S['class_names'] and n_out != len(_S['class_names']):
            print(f"  \u26a0 El modelo tiene {n_out} salidas pero hay {len(_S['class_names'])} clases")

        _S['model'] = model
        _S['history'] = None
        print(f"\n\u2714 Modelo cargado: {os.path.basename(model_path)}")
        print(f"  Input: {model.input_shape}")
        print(f"  Salidas: {n_out}")
        print(f"  Par\u00e1metros: {model.count_params():,}")
        model.summary()

w_btn_train.on_click(_entrenar)
w_btn_load.on_click(_cargar_h5)

tab_train = widgets.VBox([
    w_aug_accordion,
    w_train_accordion,
    w_btn_train,
    w_progress,
])
tab_load = widgets.VBox([
    widgets.HTML('<p>Suba o escriba la ruta a un modelo .h5 existente:</p>'),
    w_upload_h5,
    w_path_h5,
    w_btn_load,
])

w_tabs = widgets.Tab(children=[tab_train, tab_load])
w_tabs.set_title(0, 'Entrenar nuevo')
w_tabs.set_title(1, 'Cargar existente (.h5)')

display(widgets.VBox([w_tabs, w_out_train]))

---
## 3. Probar Modelo

Evalúe sobre todo el dataset o pruebe con imágenes individuales.

In [ ]:
# --- Funci\u00f3n de predicci\u00f3n reutilizable ---
def _mostrar_prediccion(img_display, img_array, titulo=''):
    """Muestra imagen + barras de confianza."""
    model = _S['model']
    class_names = _S['class_names']
    batch = np.expand_dims(img_array, 0)
    preds = model.predict(batch, verbose=0)[0]
    pred_idx = np.argmax(preds)
    conf = preds[pred_idx]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4),
                                    gridspec_kw={'width_ratios': [1, 1.2]})
    ax1.imshow(img_display)
    color = 'green' if conf > 0.7 else 'orange' if conf > 0.4 else 'red'
    ax1.set_title(f'{class_names[pred_idx]} ({conf:.1%})',
                  fontsize=14, color=color, fontweight='bold')
    ax1.axis('off')

    colors = ['#2ecc71' if i == pred_idx else '#bdc3c7'
              for i in range(len(class_names))]
    bars = ax2.barh(class_names, preds, color=colors)
    ax2.set_xlim(0, 1)
    ax2.set_xlabel('Confianza')
    for bar, val in zip(bars, preds):
        ax2.text(bar.get_width() + 0.02,
                 bar.get_y() + bar.get_height() / 2,
                 f'{val:.1%}', va='center', fontsize=10)
    if titulo:
        fig.suptitle(titulo, fontsize=11)
    plt.tight_layout()
    plt.show()
    return class_names[pred_idx], conf

In [ ]:
# --- Widgets de prueba ---
w_btn_eval = widgets.Button(description=' Evaluar dataset completo',
                            button_style='success', icon='bar-chart')
w_btn_random = widgets.Button(description=' Imagen aleatoria',
                              button_style='info', icon='random')
w_upload_img = widgets.FileUpload(accept='.jpg,.jpeg,.png', multiple=False,
                                  description='Subir foto')
w_btn_predict = widgets.Button(description=' Predecir',
                               button_style='warning', icon='eye')
w_out_test = widgets.Output()

def _check_ready():
    if _S['model'] is None:
        print("\u2718 Primero entrene o cargue un modelo (Paso 2).")
        return False
    if not _S['class_names']:
        print("\u2718 Primero cargue los datos (Paso 1).")
        return False
    return True

def _evaluar_dataset(_):
    with w_out_test:
        clear_output(wait=True)
        if not _check_ready(): return

        ddir = _S['dataset_dir']
        class_names = _S['class_names']
        img_size = _S['img_size']

        datagen = ImageDataGenerator(rescale=1./255)
        try:
            gen = datagen.flow_from_directory(
                ddir, target_size=(img_size, img_size),
                batch_size=32, class_mode='categorical',
                shuffle=False, classes=class_names
            )
        except Exception as e:
            print(f"\u2718 Error: {e}")
            return

        print(f"Evaluando {gen.samples} im\u00e1genes...")
        loss, acc = _S['model'].evaluate(gen, verbose=0)

        gen.reset()
        y_probs = _S['model'].predict(gen, verbose=0)
        y_pred = np.argmax(y_probs, axis=1)
        y_true = gen.classes
        labels = list(gen.class_indices.keys())

        print(f"\nAccuracy: {acc:.1%}")
        print(f"\nReporte:")
        print(classification_report(y_true, y_pred,
                                    target_names=labels, zero_division=0))

        cm = confusion_matrix(y_true, y_pred)
        n = len(labels)
        plt.figure(figsize=(max(5, n * 1.5), max(4, n * 1.2)))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=labels, yticklabels=labels)
        plt.xlabel('Predicci\u00f3n'); plt.ylabel('Real')
        plt.title(f'Accuracy: {acc:.1%}')
        plt.tight_layout()
        plt.show()

        # Errores
        errores = [(i, gen.filenames[i]) for i in range(len(y_true))
                   if y_pred[i] != y_true[i]]
        if errores:
            n_show = min(8, len(errores))
            print(f"\nErrores: {len(errores)} de {len(y_true)}")
            fig, axes = plt.subplots(2, 4, figsize=(16, 8))
            for ax in axes.ravel(): ax.axis('off')
            for j, ax in enumerate(axes.ravel()):
                if j >= n_show: break
                idx, fname = errores[j]
                path = os.path.join(ddir, fname)
                if os.path.exists(path):
                    ax.imshow(load_img(path, target_size=(img_size, img_size)))
                    ax.set_title(f"Real: {labels[y_true[idx]]}\n"
                                 f"Pred: {labels[y_pred[idx]]} ({y_probs[idx][y_pred[idx]]:.0%})",
                                 fontsize=9, color='red')
            plt.suptitle('Im\u00e1genes mal clasificadas', fontsize=14)
            plt.tight_layout(); plt.show()
        else:
            print("\n\u2714 Clasificaci\u00f3n perfecta.")

def _imagen_aleatoria(_):
    with w_out_test:
        clear_output(wait=True)
        if not _check_ready(): return

        ddir = _S['dataset_dir']
        img_size = _S['img_size']
        imgs = []
        for ext in ('*.jpg', '*.jpeg', '*.png'):
            imgs += glob.glob(os.path.join(ddir, '**', ext), recursive=True)
        if not imgs:
            print("No hay im\u00e1genes."); return

        path = random.choice(imgs)
        clase_real = os.path.basename(os.path.dirname(path))
        img = load_img(path, target_size=(img_size, img_size))
        arr = img_to_array(img) / 255.0
        pred, conf = _mostrar_prediccion(img, arr, titulo=f'Clase real: {clase_real}')
        ok = '\u2714' if pred == clase_real else '\u2718'
        print(f"{ok} Real: {clase_real} | Pred: {pred} ({conf:.1%})")

def _predecir_subida(_):
    with w_out_test:
        clear_output(wait=True)
        if not _check_ready(): return
        if not w_upload_img.value:
            print("\u2718 Suba una imagen primero."); return

        info = w_upload_img.value
        content = (list(info.values())[0]['content']
                   if isinstance(info, dict) else info[0].content)
        from PIL import Image
        img_size = _S['img_size']
        pil = Image.open(io.BytesIO(content)).convert('RGB').resize(
            (img_size, img_size))
        arr = np.array(pil).astype('float32') / 255.0
        pred, conf = _mostrar_prediccion(pil, arr, titulo='Imagen subida')
        print(f"Predicci\u00f3n: {pred} ({conf:.1%})")

w_btn_eval.on_click(_evaluar_dataset)
w_btn_random.on_click(_imagen_aleatoria)
w_btn_predict.on_click(_predecir_subida)

display(widgets.VBox([
    w_btn_eval,
    widgets.HTML('<hr>'),
    widgets.HBox([w_btn_random]),
    widgets.HTML('<br>'),
    widgets.HBox([w_upload_img, w_btn_predict]),
    w_out_test
]))

---
## 4. Exportar

Guarde el modelo entrenado y genere la configuración para `scripts/prueba_video.py`.

In [ ]:
w_model_name = widgets.Text(
    value='mi_modelo.h5', description='Nombre:',
    layout=widgets.Layout(width='400px')
)
w_btn_export = widgets.Button(
    description=' Guardar modelo', button_style='success', icon='download',
    layout=widgets.Layout(width='200px', height='40px')
)
w_out_export = widgets.Output()

def _exportar(_):
    with w_out_export:
        clear_output(wait=True)
        if _S['model'] is None:
            print("\u2718 No hay modelo para guardar.")
            return

        fname = w_model_name.value.strip()
        if not fname.endswith(('.h5', '.keras')):
            fname += '.h5'
        _S['model'].save(fname)
        size_mb = os.path.getsize(fname) / 1024 / 1024
        print(f"\u2714 Modelo guardado: {fname} ({size_mb:.1f} MB)")

        class_names = _S['class_names']
        img_size = _S['img_size']

        print(f"\n{'='*50}")
        print("CONFIGURACI\u00d3N PARA scripts/prueba_video.py:")
        print(f"{'='*50}")
        print(f'MODEL_PATH = \"labs/{fname}\"')
        print(f'IMG_SIZE = {img_size}')
        print(f'CLASS_NAMES = {class_names}')
        print(f'CONFIDENCE_THRESHOLD = 0.6')
        print(f"{'='*50}")

w_btn_export.on_click(_exportar)

display(widgets.VBox([w_model_name, w_btn_export, w_out_export]))